使用 经营讨论与分析 数据，计算企业数字化指标, 相关论文:


- [吴非, 胡慧芷, 林慧妍, and 任晓怡. "企业数字化转型与资本市场表现——来自股票流动性的经验证据." 管理世界 (2021).](../Paper/企业数字化转型与资本市场表...—来自股票流动性的经验证据_吴非.pdf)
- [宋德勇, 朱文博, and 丁海. "企业数字化能否促进绿色技术创新?." 财经研究 48, no. 4 (2022).](../Paper/企业数字化能否促进绿色技术...于重污染行业上市公司的考察_宋德勇.pdf)



<br>

## 一、读取数据

- **data/mda10-20.xlsx**   完整md&a数据集370M，覆盖33000+企业md&a记录。
- **data/small_mda_data.xlsx**  测试数据 small_test_data.xlsx 是从完整的数据中随机抽取的20条，结构基本一致。


以测试数据为例，方便快速实验。

In [3]:
import pandas as pd

# converters 强制声明该列为字符串，  防止股票代码 被程序识别为数字，

# 完整数据370+M 
#df = pd.read_excel('data/mda10-20.xlsx', converters={'股票代码': str})
df = pd.read_excel('data/small_mda_data.xlsx', converters={'股票代码': str})

#显示前5行
df.head()

,股票代码,公司简称,会计年度,经营讨论与分析内容
0,600689,上海三毛,2010,"（一） 管理层讨论与分析\n\n\n2010 年，面对复杂多变的外部环境，公司紧紧抓住""加快..."
1,000673,ST当代,2012,第四节 董事会报告\n本报告期，公司继续按照解决原历史遗留问题和拓展新业务并行并重的经营 方...
2,600809,山西汾酒,2013,第四节 董事会报告\n一、董事会关于公司报告期内经营情况的讨论与分析\n\n2013 年是白...
3,603969,银龙股份,2018,第四节 经营情况讨论与分析\n\n一、经营情况讨论与分析 报告期内国内外形势多变，为控制市场...
4,600229,城市传媒,2016,第四节 经营情况讨论与分析\n\n一、经营情况讨论与分析\n2016 年，公司深入学习贯彻习...


<br>

## 二、构建词典
已有词表    吴非,胡慧芷,林慧妍,任晓怡. 企业数字化转型与资本市场表现——来自股票流动性的经验证据[J]. 管理世界,2021,37(07):130-144+10.

![](img/管理世界2021吴非-企业数字化-关键词.png)

>后期，如果想自己扩展词典，可以初步筛选种子词(该篇论文的词表), 使用md&a语料文件(txt格式)， 结合cntext库的so-pmi或词向量方法，对数字化词典进行扩充。

这里我已将吴非等(2021)的词表内置到 cntext库（1.8.0版本）的 Chinese_Digitalization.pkl中。 

In [4]:
# 更新cntext至最新
!pip3 install cntext --upgrade

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/
     |████████████████████████████████| 959 kB 312 kB/s eta 0:00:01
  Attempting uninstall: cntext
    Found existing installation: cntext 1.7.9
    Uninstalling cntext-1.7.9:
      Successfully uninstalled cntext-1.7.9


In [5]:
#安装cntext库
import cntext as ct

Chinese_Digitalization_Info = ct.load_pkl_dict('Chinese_Digitalization.pkl')

print(ct.__version__)
print(Chinese_Digitalization_Info)

1.8.0
{'Referer': '吴非,胡慧芷,林慧妍,任晓怡. 企业数字化转型与资本市场表现——来自股票流动性的经验证据[J]. 管理世界,2021,37(07):130-144+10.', 'Desc': '基于这篇论文，构建了中文数字化词典，含人工智能技术、大数据技术、云计算技术、区块链技术、数字技术应用等关键词列表。 ', 'Chinese_Digitalization': {'Artificial_Intelligence': ['人工智能', '商业智能', '图像理解', '投资决策辅助系统', '智能数据分析', '智能机器人', '机器学习', '深度学习', '语义搜索', '生物识别技术', '人脸识别', '语音识别', '身份验证', '自动驾驶', '自然语言处理'], 'Big_Data': ['大数据', '数据挖掘', '文本挖掘', '数据可视化', '异构数据', '征信', '增强现实', '混合现实', '虚拟现实'], 'Cloud_Computing': ['云计算', '流计算', '图计算', '内存计算', '多方安全计算', '类脑计算', '绿色计算', '认知计算', '融合架构', '亿级并发', 'EB级存储', '物联网', '信息物理系统'], 'Block_Chains': ['区块链', '数字货币', '分布式计算', '差分隐私技术', '智能金融合约'], 'Usage_of_Digitalization': ['移动互联网', '工业互联网', '移动互联', '互联网医疗', '电子商务', '移动支付', '第三方支付', 'NFC支付', '智能能源', 'B2B', 'B2C', 'C2B', 'C2C', 'O2O', '网联', '智能穿戴', '智慧农业', '智能交通', '智能医疗', '智能客服', '智能家居', '智能投顾', '智能文旅', '智能环保', '智能电网', '智能营销', '数字营销', '无人零售', '互联网金融', '数字金融', 'Fintech', '金融科技', '量化金融', '开放银行']}}


In [6]:
#数字化相关词表
Chinese_Digitalization_Info['Chinese_Digitalization'].keys()

dict_keys(['Artificial_Intelligence', 'Big_Data', 'Cloud_Computing', 'Block_Chains', 'Usage_of_Digitalization'])

<br>

## 三、定义数字化函数


>企业数字化。目前，对于企业数字化水平的度量是相关研究的难点，现有文献主要有三种度量方法。
>- 第一，祁怀锦等（2020）使用企业年末无形资产明细项中与数字经济相关部分的金额占无形资产总额的比例度量企业数字化程度。
>- **第二，大量研究运用数字化相关关键词在年报中的词频数量或占比度量企业的数字化转型或数字化水平（赵宸宇，2021；袁淳等，2021）。**
>- 第三，相关研究采取问卷调查的方式获取企业的数字化水平数据（刘政等，2020）。


使用第二种方法，通过Python定义数字化函数，统计文本中数字化词语个数得到相应指标。

In [8]:
import cntext as ct
import pandas as pd

#数字化字典
digtal_diction = ct.load_pkl_dict('Chinese_Digitalization.pkl')['Chinese_Digitalization']

#情感分析，就是基于某种词典，对文本进行词语出现次数的统计
def digtal_function(text):
    #统计text中每类词的个数
    res = ct.sentiment(text=text,  diction=digtal_diction)
    return pd.Series(res)


test_text = '经过技术人员不懈努力， 该企业在人工智能、大数据、云计算、工业互联网等领域有了一定的市场地位....'


digtal_function(text=test_text)

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/sc/3mnt5tgs419_hk7s16gq61p80000gn/T/jieba.cache
Loading model cost 0.677 seconds.
Prefix dict has been built successfully.


Artificial_Intelligence_num     1
Big_Data_num                    1
Cloud_Computing_num             1
Block_Chains_num                0
Usage_of_Digitalization_num     1
stopword_num                   11
word_num                       24
sentence_num                    1
dtype: int64

<br>

## 四、批量计算
使用apply方法，对  [经营讨论与分析内容] 列，进行  digtal_function 运算, 得到 ``res_df``

In [11]:
#结果返回为dataframe，数字代表的是每类词出现次数
res_df = df['经营讨论与分析内容'].apply(digtal_function)
res_df.head()

,Artificial_Intelligence_num,Big_Data_num,Cloud_Computing_num,Block_Chains_num,Usage_of_Digitalization_num,stopword_num,word_num,sentence_num
0,0,0,0,0,0,504,1683,63
1,0,0,0,0,0,690,2341,43
2,0,0,0,0,2,909,3440,100
3,0,0,0,0,0,2037,10611,124
4,0,0,0,0,5,2205,8469,233


参数解读

- Artificial_Intelligence_num	 人工智能技术词出现在md&a中的次数
- Big_Data_num	 大数据技术词出现在md&a中的次数
- Cloud_Computing_num	云计算技术词出现在md&a中的次数
- Block_Chains_num	区块链技术词出现在md&a中的次数
- Usage_of_Digitalization_num	数字化应用技术词出现在md&a中的次数
- stopword_num	停用词出现在md&a中的次数
- word_num	md&a中的总词数(md&a的长度)
- sentence_num   md&a的句子数


<br>

## 五、结果整理

上一环节，将各种技术词出现次数加总，构建企业数字化词语出现个数， 并将其转为数字化指标(词频)。 

>由于这类数据具有典型的“右偏性”特征，后续在其他计量分析软件中需要将其进行**对数化处理**，从而得到刻画企业数字化转型的整体指标。

In [12]:
res_df['Digital_word_num'] = res_df[['Artificial_Intelligence_num', 'Big_Data_num', 
                                     'Cloud_Computing_num', 'Block_Chains_num', 
                                     'Usage_of_Digitalization_num']].sum(axis=1)

# [数字化相关技术词] 在 [文本总词数] 中的占比
res_df['Digital_Index'] = res_df['Digital_word_num']/res_df['word_num']
res_df.head()  

,Artificial_Intelligence_num,Big_Data_num,Cloud_Computing_num,Block_Chains_num,Usage_of_Digitalization_num,stopword_num,word_num,sentence_num,Digital_word_num,Digital_Index
0,0,0,0,0,0,504,1683,63,0,0.000000
1,0,0,0,0,0,690,2341,43,0,0.000000
2,0,0,0,0,2,909,3440,100,2,0.000581
3,0,0,0,0,0,2037,10611,124,0,0.000000
4,0,0,0,0,5,2205,8469,233,5,0.000590


<br>

## 六、保存结果

合并df 和 ``res_df``, 将其存储到新的文件中。

In [13]:
df2 = pd.concat([df, res_df], axis=1)

df2.head()

,股票代码,公司简称,会计年度,经营讨论与分析内容,Artificial_Intelligence_num,Big_Data_num,Cloud_Computing_num,Block_Chains_num,Usage_of_Digitalization_num,stopword_num,word_num,sentence_num,Digital_word_num,Digital_Index
0,600689,上海三毛,2010,"（一） 管理层讨论与分析\n\n\n2010 年，面对复杂多变的外部环境，公司紧紧抓住""加快...",0,0,0,0,0,504,1683,63,0,0.000000
1,000673,ST当代,2012,第四节 董事会报告\n本报告期，公司继续按照解决原历史遗留问题和拓展新业务并行并重的经营 方...,0,0,0,0,0,690,2341,43,0,0.000000
2,600809,山西汾酒,2013,第四节 董事会报告\n一、董事会关于公司报告期内经营情况的讨论与分析\n\n2013 年是白...,0,0,0,0,2,909,3440,100,2,0.000581
3,603969,银龙股份,2018,第四节 经营情况讨论与分析\n\n一、经营情况讨论与分析 报告期内国内外形势多变，为控制市场...,0,0,0,0,0,2037,10611,124,0,0.000000
4,600229,城市传媒,2016,第四节 经营情况讨论与分析\n\n一、经营情况讨论与分析\n2016 年，公司深入学习贯彻习...,0,0,0,0,5,2205,8469,233,5,0.000590


In [16]:
df2[['股票代码', '公司简称', '会计年度', 'Digital_Index']].to_excel('output/corporate_digitalization.xlsx', index=False)

In [18]:
import pandas as pd

df = pd.read_excel('output/corporate_digitalization.xlsx', converters={'股票代码': str})
df

,股票代码,公司简称,会计年度,Digital_Index
0,600689,上海三毛,2010,0.000000
1,000673,ST当代,2012,0.000000
2,600809,山西汾酒,2013,0.000581
3,603969,银龙股份,2018,0.000000
4,600229,城市传媒,2016,0.000590
5,600191,华资实业,2016,0.000000
6,600581,八一钢铁,2010,0.000000
7,002523,天桥起重,2017,0.000557
8,002164,宁波东力,2020,0.000271
9,300615,欣天科技,2019,0.000000
